第二个笔记主要用于检查 `data/final_monthly/` 中拟用于建模的序列是否存在单位根。使用 ADF 检验，并用 KPSS 补充判断，为选择变量形式和降低伪回归风险提供依据。检验沿用第一个笔记的 2012-2024 年训练期，不使用后续留出的预测评估期。主要月度变量已经转换为对数变化率，因此先检验当前对数变化的形式。如果两种检验及敏感性检查共同提示非平稳，再对候选序列进行额外一阶差分诊断并重新检验；是否用于模型仍需结合季节性、结构变化和经济含义判断。

The second notebook checks for unit roots in the series intended for modelling from `data/final_monthly/`. ADF tests are complemented by KPSS tests to guide the choice of variable transformations and reduce the risk of spurious regression. The tests use the same 2012-2024 training period as the first notebook and exclude the held-out forecast evaluation period. Most monthly variables are already log changes, so their current form is tested before considering any further differencing. If both tests and the sensitivity checks indicate nonstationarity, an additional first difference is examined and tested again. Its eventual use in the model also depends on seasonality, structural changes, and economic interpretation.

In [18]:
# import packages
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels
from IPython.display import display
from statsmodels.tools.sm_exceptions import InterpolationWarning
from statsmodels.tsa.stattools import adfuller, kpss

下面读取 `data/final_monthly/` 中全部 9个 CSV 的数值变量，包括 7 条月度经济序列、2 条年度变化序列。

两项年度指标按年份各取一个值，使用 2012 至 2024 年的 13 个观测进行年频检验。政策事件变量全部列入最终结果，标记为常规 ADF/KPSS 不适用，并保留原有编码。

The following cell reads the numeric variables from all ten CSVs in `data/final_monthly/`: seven monthly economic series, two annual change series.

Each annual series contributes one value per year, giving 13 observations from 2012 to 2024 for annual-frequency tests. Every policy dummy is included in the final results, marked as outside the scope of standard ADF/KPSS tests, and retains its existing coding.

In [19]:
# dataset file path
DATASET_PATH = Path("data/final_monthly")
if not DATASET_PATH.is_dir():
    DATASET_PATH = Path("../data/final_monthly")
if not DATASET_PATH.is_dir():
    raise FileNotFoundError("Run from the project root or notebook directory.")

dataset_name_list = [
    "base_money_log_change_monthly.csv",
    "current_account_balance_annual_change_pp_monthly.csv",
    "food_imports_log_change_monthly.csv",
    "fuel_imports_log_change_monthly.csv",
    "fx_reserves_log_change_monthly.csv",
    "general_government_gross_debt_annual_change_pp_monthly.csv",
    "iron_ore_exports_log_change_monthly.csv",
    "NO_FILL_iron_ore_exports_log_change_monthly.csv",
    "USDSLE_monthly.csv",
    "Sierra_Leone_Policy_Event_Dummies_monthly.csv",
]

检验使用常数项设定（`regression="c"`）。月度 ADF 在最多 12 阶增广滞后中按 AIC 选阶，并用 BIC 检查敏感性；若选到上界，再将候选上限扩大到 18 阶。月度 KPSS 使用自动滞后，并以固定 12 阶作对照。

年度样本只有 13 个观测，ADF 上限设为 1 阶，必要时扩至 2 阶；KPSS 使用自动滞后和固定 1 阶。年频滞后的单位是年。年度结果作为探索性证据，不据此自动追加差分。显著性水平为 0.05，各变量的 p 值按单项检验解释。

The tests include a constant (`regression="c"`). Monthly ADF selects lags by AIC up to a maximum of 12, with BIC as a sensitivity check. The maximum increases to 18 if the selected lag reaches the initial limit. Monthly KPSS uses automatic lags and a fixed 12-lag comparison.

The annual samples contain 13 observations. Annual ADF uses a maximum of one lag, extended to two when needed; KPSS uses automatic lags and a fixed one-lag comparison. These lags represent years. Annual results are exploratory and do not trigger automatic additional differencing. The significance level is 0.05, and p-values are interpreted separately for each variable.

In [20]:
# set constants incl. dates and parameters for statistical tests
TRAIN_START = "2012-01-31"
TRAIN_END = "2024-12-31"
ALPHA = 0.05
REGRESSION = "c"
MAX_LAG = 12
EXTENDED_MAX_LAG = 18
KPSS_FIXED_LAGS = 12
ANNUAL_MAX_LAG = 1
ANNUAL_EXTENDED_MAX_LAG = 2
ANNUAL_KPSS_FIXED_LAGS = 1

In [21]:
def longest_complete_segment(values):
    groups = values.isna().cumsum()
    segments = [
        part.dropna() for _, part in values.groupby(groups) if part.notna().any()
    ]
    return max(segments, key=len) if segments else values.iloc[:0]

本轮对 7 条月度序列和 2 条年频序列执行检验。其中，月度 `NO_FILL` 铁矿石沿用 2012-01 至 2018-05 的 77 个月连续样本，其结果不覆盖另外 47 个月。年度指标各使用 13 年数据。40 个事件虚拟变量在最终汇总中逐项列出，不进行额外差分。

ADF 的原假设是存在单位根；KPSS 的原假设是围绕常数水平平稳。对同一样本，按下表解释结果：

| ADF | KPSS | 结果标签 | 解释 |
|---|---|---|---|
| 拒绝单位根 | 不拒绝平稳 | supported_stationarity | 支持保留当前形式 |
| 不拒绝单位根 | 拒绝平稳 | nonstationarity_signal | 核对敏感性结果后考虑差分诊断 |
| 拒绝单位根 | 拒绝平稳 | conflicting_tests | 检验结果冲突，需要进一步判断 |
| 不拒绝单位根 | 不拒绝平稳 | inconclusive | 证据不足，不自动差分 |


This run tests seven monthly series and two annual series. The monthly `NO_FILL` iron ore sample remains January 2012 to May 2018, with 77 consecutive observations; its findings do not cover the other 47 months. Each annual series uses 13 years. All 40 event dummies appear individually in the final summary and are not additionally differenced.

The null hypothesis of the ADF test is that a unit root exists; the null hypothesis of the KPSS test is stationarity around a constant level. For the same sample, interpret the results as follows:
| ADF | KPSS | Result Label | Interpretation |
|---|---|---|---|
| Reject unit root | Do not reject stationarity | supported_stationarity | Supports retaining the current form |
| Do not reject unit root | Reject stationarity | nonstationarity_signal | Consider differencing diagnostics after checking sensitivity results |
| Reject unit root | Reject stationarity | conflicting_tests | Test results conflict; further judgment is needed |
| Do not reject unit root | Do not reject stationarity | inconclusive | Insufficient evidence; do not automatically difference |

ADF tests a unit-root null, while KPSS tests a level-stationarity null. Rejecting only the ADF null supports retaining the current form. Rejecting only the KPSS null gives a nonstationarity signal requiring sensitivity checks. Rejecting both yields conflicting evidence; rejecting neither is inconclusive. 

In [22]:
def interpret_tests(adf_p, kpss_p):
    adf_reject = adf_p < ALPHA
    kpss_reject = kpss_p < ALPHA
    if adf_reject and not kpss_reject:
        return "supported_stationarity"
    if not adf_reject and kpss_reject:
        return "nonstationarity_signal"
    return "conflicting_tests" if adf_reject else "inconclusive"


def kpss_p_label(result):
    # Outside the lookup table, the returned p-value is a bound.
    if result.statistic < result.critical_values["10%"]:
        return "> 0.10"
    if result.statistic > result.critical_values["1%"]:
        return "< 0.01"
    return f"{result.pvalue:.4f}"


def run_unit_root_tests(values, frequency="monthly"):
    annual = frequency == "annual"
    requested_max = ANNUAL_MAX_LAG if annual else MAX_LAG
    requested_extended = ANNUAL_EXTENDED_MAX_LAG if annual else EXTENDED_MAX_LAG
    fixed_kpss_lags = ANNUAL_KPSS_FIXED_LAGS if annual else KPSS_FIXED_LAGS

    # Lag limits depend on the sample length and deterministic terms.
    n_deterministic = {"c": 1, "ct": 2}[REGRESSION]
    feasible_max = len(values) // 2 - n_deterministic - 1
    maxlag = min(requested_max, feasible_max)
    adf_aic = adfuller(
        values, maxlag=maxlag, regression=REGRESSION,
        autolag="AIC", result_object=True,
    )
    adf_bic = adfuller(
        values, maxlag=maxlag, regression=REGRESSION,
        autolag="BIC", result_object=True,
    )

    extended = None
    extended_max = min(requested_extended, feasible_max)
    if adf_aic.lags == maxlag and extended_max > maxlag:
        extended = adfuller(
            values, maxlag=extended_max, regression=REGRESSION,
            autolag="AIC", result_object=True,
        )

    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", InterpolationWarning)
        kp_auto = kpss(
            values, regression=REGRESSION, nlags="auto", result_object=True,
        )
        kp_fixed = kpss(
            values, regression=REGRESSION,
            nlags=fixed_kpss_lags, result_object=True,
        )
    # Expected table-bound warnings are represented by > / < labels below.
    # Keep all warning messages in the full result, and re-emit other warnings.
    for item in caught:
        if not issubclass(item.category, InterpolationWarning):
            warnings.warn(str(item.message), item.category, stacklevel=2)

    baseline = interpret_tests(adf_aic.pvalue, kp_auto.pvalue)
    adf_sensitive = (adf_aic.pvalue < ALPHA) != (adf_bic.pvalue < ALPHA)
    if extended is not None:
        adf_sensitive |= (adf_aic.pvalue < ALPHA) != (extended.pvalue < ALPHA)
    kpss_sensitive = (kp_auto.pvalue < ALPHA) != (kp_fixed.pvalue < ALPHA)
    at_limit = (
        extended.lags == extended_max
        if extended is not None else adf_aic.lags == maxlag
    )

    return {
        "frequency": frequency,
        "test_start": str(values.index[0]),
        "test_end": str(values.index[-1]),
        "input_n": len(values),
        "adf_stat": adf_aic.statistic,
        "adf_p": adf_aic.pvalue,
        "adf_lag": adf_aic.lags,
        "adf_nobs": adf_aic.nobs,
        "adf_critical_5pct": adf_aic.critical_values["5%"],
        "adf_maxlag": maxlag,
        "adf_bic_p": adf_bic.pvalue,
        "adf_bic_lag": adf_bic.lags,
        "adf_extended_maxlag": extended_max if extended is not None else np.nan,
        "adf_extended_p": extended.pvalue if extended is not None else np.nan,
        "adf_extended_lag": extended.lags if extended is not None else np.nan,
        "kpss_stat": kp_auto.statistic,
        "kpss_p": kp_auto.pvalue,
        "kpss_p_bound": kpss_p_label(kp_auto),
        "kpss_lag": kp_auto.lags,
        "kpss_critical_5pct": kp_auto.critical_values["5%"],
        "kpss_fixed_stat": kp_fixed.statistic,
        "kpss_fixed_p": kp_fixed.pvalue,
        "kpss_fixed_p_bound": kpss_p_label(kp_fixed),
        "kpss_fixed_lag": kp_fixed.lags,
        "joint_result": baseline,
        "sensitivity_flag": bool(adf_sensitive or kpss_sensitive),
        "lag_limit_unresolved": bool(at_limit),
        "warnings": " | ".join(dict.fromkeys(str(item.message).strip() for item in caught)),
    }

KPSS 的 p 值来自查表。输出 `> 0.10` 或 `< 0.01` 时应按界限解读。原始数值、5% 临界值及警告信息保留在 `unit_root_results` 中。`sensitivity_flag=True` 表示拒绝／不拒绝的结论随选阶设定改变；`lag_limit_unresolved=True` 表示扩大上限后仍需检查选阶。

KPSS p-values come from a lookup table. Values shown as `> 0.10` or `< 0.01` are bounds. Numeric returns, 5% critical values, and warning messages are retained in `unit_root_results`. A true sensitivity flag means that a rejection decision changes with the lag specification. A true unresolved-limit flag means the lag selection still needs review after considering a higher maximum.

In [23]:
# Build test inputs here before using them; rerunning this cell refreshes the samples.
series_data = {}
test_segments = {}
variable_rows = []

for filename in dataset_name_list:
    frame = pd.read_csv(
        DATASET_PATH / filename, index_col=0, parse_dates=[0],
    ).loc[TRAIN_START:TRAIN_END]

    # Numeric columns include every event dummy; text metadata stays out.
    for name in frame.select_dtypes(include="number").columns:
        values = frame[name].astype(float)
        series_data[name] = values

        if filename == "Sierra_Leone_Policy_Event_Dummies_monthly.csv":
            variable_type = "policy_dummy"
            frequency = "monthly"
            sample = values
            note = "event indicator; ADF/KPSS not applicable"
        elif "_annual_change_pp_monthly" in filename:
            variable_type = "annual_change"
            frequency = "annual"
            sample = values.groupby(values.index.to_period("Y")).first()
            note = "one annual change per year; exploratory inference"
            test_segments[name] = sample
        else:
            variable_type = "monthly_change"
            frequency = "monthly"
            sample = longest_complete_segment(values)
            note = "longest complete training interval"
            if "iron_ore" in filename:
                note += "; earlier fills remain" if filename.startswith("NO_FILL") else "; source fills retained"
            test_segments[name] = sample

        variable_rows.append({
            "variable": name,
            "source_file": filename,
            "variable_type": variable_type,
            "frequency": frequency,
            "sample_start": str(sample.index[0]),
            "sample_end": str(sample.index[-1]),
            "sample_n": len(sample),
            "sample_note": note,
        })

variable_info = pd.DataFrame(variable_rows).set_index("variable")
print(f"Loaded {len(series_data)} variables from {len(dataset_name_list)} CSVs.")
print(variable_info["variable_type"].value_counts().to_string())
print(f"Prepared {len(test_segments)} economic series for ADF/KPSS.")

unit_root_results = pd.DataFrame.from_dict(
    {
        name: run_unit_root_tests(values, variable_info.loc[name, "frequency"])
        for name, values in test_segments.items()
    },
    orient="index",
).rename_axis("variable")

baseline_columns = [
    "frequency", "input_n", "adf_stat", "adf_p", "adf_lag", "adf_nobs",
    "kpss_stat", "kpss_p_bound", "kpss_lag", "joint_result",
]
sensitivity_columns = [
    "adf_bic_p", "adf_bic_lag", "kpss_fixed_stat",
    "kpss_fixed_p_bound", "kpss_fixed_lag",
    "sensitivity_flag", "lag_limit_unresolved",
]
with pd.option_context(
    "display.max_columns", None,
    "display.float_format", lambda value: f"{value:.6g}",
):
    print("Baseline tests")
    display(unit_root_results[baseline_columns])
    print("Sensitivity checks")
    display(unit_root_results[sensitivity_columns])
    extended_rows = unit_root_results["adf_extended_p"].notna()
    if extended_rows.any():
        print("ADF upper-limit checks")
        display(unit_root_results.loc[extended_rows, [
            "adf_maxlag", "adf_lag", "adf_p",
            "adf_extended_maxlag", "adf_extended_lag", "adf_extended_p",
        ]])

Loaded 49 variables from 10 CSVs.
variable_type
policy_dummy      40
monthly_change     7
annual_change      2
Prepared 9 economic series for ADF/KPSS.
Baseline tests


,frequency,input_n,adf_stat,adf_p,adf_lag,adf_nobs,kpss_stat,kpss_p_bound,kpss_lag,joint_result
variable,,,,,,,,,,
base_money_log_change_monthly,monthly,156,-8.12258,1.1455e-12,2,153,0.15341,> 0.10,8,supported_stationarity
current_account_balance_annual_change_pp_monthly,annual,13,-4.33575,0.000384985,0,12,0.352737,0.0975,0,supported_stationarity
food_imports_log_change_monthly,monthly,156,-7.61625,2.18787e-11,7,148,0.0355886,> 0.10,9,supported_stationarity
fuel_imports_log_change_monthly,monthly,156,-5.5029,2.05226e-06,12,143,0.243842,> 0.10,81,supported_stationarity
fx_reserves_log_change_monthly,monthly,156,-11.2616,1.61718e-20,0,155,0.20318,> 0.10,0,supported_stationarity
general_government_gross_debt_annual_change_pp_monthly,annual,13,-2.82413,0.054909,0,12,0.172444,> 0.10,0,inconclusive
iron_ore_exports_log_change_monthly,monthly,156,-18.3471,2.24025e-30,0,155,0.138618,> 0.10,19,supported_stationarity
NO_FILL_iron_ore_exports_log_change_monthly,monthly,77,-7.99507,2.41654e-12,1,75,0.26266,> 0.10,7,supported_stationarity
USDSLE_monthly,monthly,156,-14.1584,2.1019e-26,0,155,0.243441,> 0.10,2,supported_stationarity


Sensitivity checks


,adf_bic_p,adf_bic_lag,kpss_fixed_stat,kpss_fixed_p_bound,kpss_fixed_lag,sensitivity_flag,lag_limit_unresolved
variable,,,,,,,
base_money_log_change_monthly,1.27651e-24,0,0.192519,> 0.10,12,False,False
current_account_balance_annual_change_pp_monthly,0.000384985,0,0.355936,0.0961,1,False,False
food_imports_log_change_monthly,8.68233e-29,1,0.0452454,> 0.10,12,False,False
fuel_imports_log_change_monthly,2.34672e-20,2,0.0799465,> 0.10,12,False,False
fx_reserves_log_change_monthly,1.61718e-20,0,0.188298,> 0.10,12,False,False
general_government_gross_debt_annual_change_pp_monthly,0.054909,0,0.158409,> 0.10,1,False,False
iron_ore_exports_log_change_monthly,2.24025e-30,0,0.137838,> 0.10,12,False,False
NO_FILL_iron_ore_exports_log_change_monthly,1.40922e-25,0,0.252192,> 0.10,12,False,False
USDSLE_monthly,2.1019e-26,0,0.186139,> 0.10,12,False,False


ADF upper-limit checks


,adf_maxlag,adf_lag,adf_p,adf_extended_maxlag,adf_extended_lag,adf_extended_p
variable,,,,,,
fuel_imports_log_change_monthly,12,12,2.05226e-06,18,12,2.05226e-06


5 条月度序列的 ADF 都拒绝单位根，ADF的p值小于0.05；KPSS 都未拒绝水平平稳，实际 KPSS p 值均大于 0.10。ADF 改用 BIC、KPSS 改用固定 12 阶后，判断没有改变。因此此类月度数据可以继续进行下一步处理。

剩余2条年度数据序列的ADF与KPSS结果如下：

| 年度百分点变化 | ADF p 值 | KPSS p 值（自动滞后） | 解释 |
|---|---:|---:|---|
| 经常账户/GDP | 0.000385 | 0.097527 | 支持保留当前年度变化形式 |
| 政府债务/GDP | 0.054909 | > 0.10 | 两项检验均未拒绝各自原假设，结论不明确 |

政府债务/GDP序列的ADF检验p值大于0.05，而KPSS统计量大于0.1。鉴于数据缺失的特殊性，ADF检验p值大于0.05尚不足以证明该序列存在非平稳性；现有证据也无法对此作出明确判断，且缺乏充分依据支持再进行差分。因此，后续分析将继续进行。

All 5 monthly series reject the unit root under ADF, with ADF p-values less than 0.05; under KPSS, none reject level stationarity, and the actual KPSS p-values are all greater than 0.10. After switching ADF to BIC and KPSS to a fixed lag of 12, the conclusions remained unchanged. Therefore, this type of monthly data can proceed to the next step of processing.

The ADF and KPSS results for the remaining 2 annual data series are as follows:

| Annual percentage-point change | ADF p-value | KPSS p-value (automatic lag) | Interpretation |
|---|---:|---:|---|
| Current account/GDP | 0.000385 | 0.097527 | Supports retaining the current annual-change specification |
| Government debt/GDP | 0.054909 | > 0.10 | Neither test rejects its respective null hypothesis; the conclusion is inconclusive |

For the government debt/GDP series, the ADF test p-value is greater than 0.05, while the KPSS statistic is greater than 0.1. Given the particular nature of the missing data, an ADF test p-value greater than 0.05 is not sufficient to prove that the series is nonstationary; the existing evidence also cannot provide a clear judgment on this matter, and there is insufficient basis to support further differencing. Therefore, the subsequent analysis will continue.